In [1]:
from kubeflow.trainer import TrainerClient, CustomTrainer
from kubeflow.trainer import options as k8s_options
from kubeflow.trainer.backends import kubernetes

In [2]:
def run_job():
    import subprocess

    subprocess.run(["bash", "run.sh"], cwd="/app/deploy/innuce-HPP", check=True)

In [ ]:
VOLUME_NAME="workspace-demo-0" # Put your volume name
SUBPATH="demo/RockScissorPaper" # Put your subpath where the repo is located inside your volume

job_patch = k8s_options.RuntimePatch(
    training_runtime_spec=k8s_options.TrainingRuntimeSpecPatch(
        template=k8s_options.JobSetTemplatePatch(
            spec=k8s_options.JobSetSpecPatch(
                replicated_jobs=[
                    k8s_options.ReplicatedJobPatch(
                        name="node",
                        template=k8s_options.JobTemplatePatch(
                            spec=k8s_options.JobSpecPatch(
                                template=k8s_options.PodTemplatePatch(
                                    spec=k8s_options.PodSpecPatch(
                                        volumes=[
                                            {
                                                "name": VOLUME_NAME,
                                                "persistentVolumeClaim": {"claimName": VOLUME_NAME},
                                            }
                                        ],
                                        containers=[
                                            k8s_options.ContainerPatch(
                                                name="node",  # use the container name in your runtime/pod
                                                volume_mounts=[
                                                    {
                                                        "name": VOLUME_NAME,
                                                        "mountPath": "/app",
                                                        "subPath": SUBPATH,
                                                    }
                                                ],
                                            )
                                        ],
                                    )
                                )
                            )
                        ),
                    )
                ]
            )
        )
    )
)

In [5]:
client = TrainerClient()

job_id = client.train(
    runtime="akida-runtime", 
    trainer=CustomTrainer(
        func=run_job,
        image="pepperide/akida-runtime:2.13.0-py311",
        packages_to_install=[],
        num_nodes=1,
        resources_per_node={
            "cpu": 2,
            "memory": "8Gi",
            "brainchip.com/akida1000": 1
        },
    ),
    options=[job_patch,],
)
print(job_id)

a5965ffeb3f9
